In [1]:
!pip install -q sentence-transformers underthesea pandas numpy

In [2]:
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize

In [3]:
JSON_FILE = "LuatLaoDong2019_tree.json"

with open(JSON_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

nodes = data["nodes"]

print("Document:", data["metadata"]["document_name"])
print("Version:", data["metadata"]["version"])
print("Total nodes:", len(nodes))

Document: LuatLaoDong2019
Version: 1.0
Total nodes: 1169


In [4]:
corpus = []

for node in nodes:

    content = node.get("content", "").strip()

    if not content:
        continue

    corpus.append({
        "node_id": node["node_id"],
        "content": content,
        "path": node.get("path", [])
    })

print("Nodes with content:", len(corpus))

Nodes with content: 998


In [5]:
def get_path(node):

    path_parts = []

    for p in node["path"]:

        if p["type"] == "CHUONG":
            path_parts.append(
                f"Chương {p['number']}"
            )

        elif p["type"] == "DIEU":
            path_parts.append(
                f"Điều {p['number']}"
            )

        elif p["type"] == "KHOAN":
            path_parts.append(
                f"Khoản {p['number']}"
            )

        elif p["type"] == "DIEM":
            path_parts.append(
                f"Điểm {p['number']}"
            )

    return " ".join(path_parts)

In [6]:
for item in corpus[:5]:

    print("=" * 80)
    print("Node ID :", item["node_id"])
    print("Path    :", get_path(item))
    print("Content :", item["content"])

Node ID : 1
Path    : Chương I
Content : NHỮNG QUY ĐỊNH CHUNG
Node ID : 2
Path    : Chương I Điều 1
Content : Bộ luật Lao động quy định tiêu chuẩn lao động; quyền, nghĩa vụ, trách nhiệm của người lao động, người sử dụng lao động, tổ chức đại diện người lao động tại cơ sở, tổ chức đại diện người sử dụng lao động trong quan hệ lao động và các quan hệ khác liên quan trực tiếp đến quan hệ lao động; quản lý nhà nước về lao động.
Node ID : 4
Path    : Chương I Điều 2 Khoản 1
Content : Người lao động, người học nghề, người tập nghề và người làm việc không có quan hệ lao động.
Node ID : 5
Path    : Chương I Điều 2 Khoản 2
Content : Người sử dụng lao động.
Node ID : 6
Path    : Chương I Điều 2 Khoản 3
Content : Người lao động nước ngoài làm việc tại Việt Nam.


In [7]:
MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully.


In [8]:
def preprocess_text(text):
    return word_tokenize(
        text,
        format="text"
    )

In [9]:
documents = [
    preprocess_text(item["content"])
    for item in corpus
]

print("Total documents:", len(documents))

Total documents: 998


In [10]:
document_embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

document_embeddings = np.asarray(
    document_embeddings
)

print(
    "Embedding shape:",
    document_embeddings.shape
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embedding shape: (998, 768)


In [11]:
def retrieve(query, top_k=5):

    query_processed = preprocess_text(query)

    query_embedding = model.encode(
        [query_processed],
        normalize_embeddings=True
    )[0]
    
    scores = np.dot(
        document_embeddings,
        query_embedding
    )

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        item = corpus[idx]

        results.append({
            "Rank": rank,
            "Score": round(
                float(scores[idx]),
                4
            ),
            "Path": get_path(item),
            "Content": item["content"]
        })

    return results

In [12]:
def retrieve_dataframe(query, top_k=5):

    results = retrieve(
        query=query,
        top_k=top_k
    )

    return pd.DataFrame(results)

In [13]:
query = "Nam gới được nghỉ hưu bao nhiêu tuổi?"
TOP_K = 5

results_df = retrieve_dataframe(
    query,
    top_k=TOP_K
)

display(results_df)

,Rank,Score,Path,Content
0,1,0.6391,Chương XII Điều 169 Khoản 2,Tuổi nghỉ hưu của người lao động trong điều ki...
1,2,0.4055,Chương XII Điều 169 Khoản 3,Người lao động bị suy giảm khả năng lao động; ...
2,3,0.4052,Chương XVII Điều 219 Khoản 3,"Lao động nữ là cán bộ, công chức cấp xã hoặc l..."
3,4,0.3923,Chương III Điều 35 Khoản 2 Điểm e,Đủ tuổi nghỉ hưu theo quy định tại Điều 169 củ...
4,5,0.3921,Chương III Điều 18 Khoản 4 Điểm a,Người lao động từ đủ 18 tuổi trở lên;
